In [9]:
import os

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

!pwd
os.chdir("../RecSys_Course_AT_PoliMi")
!pwd

#!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi


In [10]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from scipy.sparse import csr_matrix
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt

from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
#from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

#from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender

# from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

#from Recommenders.hybrid.m2_model import IntegratedHierarchicalHybridRecommender
from Recommenders.hybrid.SimilarityMergingHybridRecommender import SimilarityMergingHybridRecommender
from Recommenders.hybrid.LinearHybridRecommender import GeneralizedLinearCoupleHybridRecommender


In [11]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

# Training knn costum similarity

In [12]:
SLIM_params = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

KNN_params = {
    'similarity': 'tversky',
    'topK': 8,
    'shrink': 100,
    'tversky_alpha': 0.18445514996044549,
    'tversky_beta': 1.7490566752549062,
    'feature_weighting': 'TF-IDF',
}

"""IALS_params = {
    'iterations': 135,
    'factors': 87,
    'alpha': 7.762338288061237,
    'regularization': 0.004799745261257595
}"""

EASE_params = {
    'topK': 1431,
    'l2_norm': 426.57622242296605
}

"""rp3_params = {
    'alpha': 0.7733352330682174,
    'beta': 0.4139018623121251,
    'topK': 35
}"""

"rp3_params = {\n    'alpha': 0.7733352330682174,\n    'beta': 0.4139018623121251,\n    'topK': 35\n}"

In [13]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))


In [14]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [15]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [16]:
import os
from scipy import sparse

output_folder = "./saved_models/"
urm_folder = "./saved_urm/"

for folder in [output_folder, urm_folder]:
    if not os.path.exists(folder):
        os.makedirs(folder)

prefitted_folds = []

for i in range(5):
    print(f"Fitting fold {i+1}/5...")
    
    urm_path = os.path.join(urm_folder, f"URM_train_fold_{i}.npz")
    
    # 1. Gestione URM Train
    if os.path.exists(urm_path):
        print(f"Loading URM_train for fold {i}...")
        URM_train = sparse.load_npz(urm_path)
    else:
        print(f"Creating and saving URM_train for fold {i}...")
        URM_train = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        sparse.save_npz(urm_path, URM_train)

    URM_test = URM_parts[i]
    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[20])

    # 2. Inizializzazione Modelli
    recommender_ease = EASE_R_Recommender(URM_train)
    recommender_slim = SLIMElasticNetRecommender(URM_train)
    recommender_knn = ItemKNNCFRecommender(URM_train)

    model_names = {
        "ease": (recommender_ease, EASE_params),
        "slim": (recommender_slim, SLIM_params),
        "knn": (recommender_knn, KNN_params)
    }

    # 3. Fit o Load dei modelli
    for name, (model, params) in model_names.items():
        file_name = f"{name}_fold_{i}"
        # Verifichiamo se il file del modello esiste (il framework aggiunge solitamente un'estensione o crea una cartella)
        try:
            model.load_model(output_folder, file_name=file_name)
            print(f"Loaded {name} from disk.")
        except (FileNotFoundError, Exception):
            print(f"Fitting {name}...")
            model.fit(**params)
            model.save_model(output_folder, file_name=file_name)
            print(f"Saved {name} to disk.")
        #print(type(recommender_slim.W_sparse), recommender_slim.W_sparse.nnz)

    # 4. Popolamento lista per ottimizzazione/test
    fold_data = {
        "URM_train": URM_train,
        "ease": recommender_ease,
        "slim": recommender_slim,
        "knn": recommender_knn,
        "evaluator": evaluator_test
    }
    
    prefitted_folds.append(fold_data)

print("\nTask completato: tutti i fold sono pronti in memoria.")

print("Pre-training completato.")

Fitting fold 1/5...
Creating and saving URM_train for fold 0...
EvaluatorHoldout: Ignoring 32 ( 0.1%) Users that have less than 1 test interactions
EASE_R_Recommender: Loading model from file './saved_models/ease_fold_0'
Fitting ease...
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 16.23 sec
EASE_R_Recommender: Saving model in file './saved_models/ease_fold_0'
EASE_R_Recommender: Saving complete
Saved ease to disk.
SLIMElasticNetRecommender: Loading model from file './saved_models/slim_fold_0'
Fitting slim...
SLIMElasticNetRecommender: Processed 4797 (68.8%) in 5.00 min. Items per second: 15.99
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.26 min. Items per second: 16.00
SLIMElasticNetRecommender: Saving model in file './saved_models/slim_fold_0'
SLIMElasticNetRecommender: Saving complete
Saved slim to disk.
ItemKNNCFRecommender: Loading model from file './saved_models/knn_fold_0'
Fitting knn...
Unable to load Cython Compute_Similarity, re

FileNotFoundError: [Errno 2] No such file or directory: './saved_urm/URM_train_fold_4.npz'

In [ ]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

    start_time = time.time()
    scores = []
    for fold_data in prefitted_folds:
        
        # Recuperiamo i modelli e i dati dal dizionario
        URM_train = fold_data["URM_train"]
        recommender_ease = fold_data["ease"]
        recommender_SLIM = fold_data["slim"]
        recommender_knn = fold_data["knn"]
        evaluator_test = fold_data["evaluator"]
        
        
        recommender = SimilarityMergingHybridRecommender(
            URM_train, 
            recommender_ease,
            recommender_SLIM, 
            recommender_knn
        )
        
        alpha=optuna_trial.suggest_float("alpha", 0.0, 1)
        beta=optuna_trial.suggest_float("beta", 0.0, 1)
        
        recommender.fit(alpha, beta)
        
        result, _ = evaluator_test.evaluateRecommender(recommender)
        #print("prova = ", result["MAP"].values[0])
        #print(i)
        #print(result["RECALL"])
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [ ]:
import optuna

optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 200)

[I 2025-12-26 18:03:04,301] A new study created in memory with name: no-name-a377d277-f027-44e6-8161-4eeb41f56a8e


EvaluatorHoldout: Processed 27072 (100.0%) in 12.27 sec. Users per second: 2206
EvaluatorHoldout: Processed 27055 (100.0%) in 13.24 sec. Users per second: 2043
EvaluatorHoldout: Processed 27059 (100.0%) in 12.30 sec. Users per second: 2201
EvaluatorHoldout: Processed 27065 (100.0%) in 12.39 sec. Users per second: 2185
EvaluatorHoldout: Processed 27057 (100.0%) in 12.56 sec. Users per second: 2155


[I 2025-12-26 18:04:11,905] Trial 0 finished with value: 0.2600658108245092 and parameters: {'alpha': 0.8080623331177008, 'beta': 0.7792411478219032}. Best is trial 0 with value: 0.2600658108245092.


[0.258857409325813, 0.26003079405360396, 0.2606946040121549, 0.2600487905650703, 0.2606974561659036]
EvaluatorHoldout: Processed 27072 (100.0%) in 12.75 sec. Users per second: 2123
EvaluatorHoldout: Processed 27055 (100.0%) in 12.53 sec. Users per second: 2158
EvaluatorHoldout: Processed 27059 (100.0%) in 12.30 sec. Users per second: 2200
EvaluatorHoldout: Processed 27065 (100.0%) in 12.41 sec. Users per second: 2181
EvaluatorHoldout: Processed 27057 (100.0%) in 12.35 sec. Users per second: 2191


[I 2025-12-26 18:05:17,361] Trial 1 finished with value: 0.2776467353157343 and parameters: {'alpha': 0.7116830318014558, 'beta': 0.33295110148828744}. Best is trial 1 with value: 0.2776467353157343.


...

[0.287158312483761, 0.2880623258537209, 0.28900101934505795, 0.28894917520826807, 0.2901211846997007]
EvaluatorHoldout: Processed 27072 (100.0%) in 11.93 sec. Users per second: 2269
EvaluatorHoldout: Processed 27055 (100.0%) in 11.95 sec. Users per second: 2264
EvaluatorHoldout: Processed 27059 (100.0%) in 11.85 sec. Users per second: 2284
EvaluatorHoldout: Processed 27065 (100.0%) in 11.98 sec. Users per second: 2260
EvaluatorHoldout: Processed 27057 (100.0%) in 11.96 sec. Users per second: 2263


[I 2025-12-26 21:30:44,948] Trial 198 finished with value: 0.28830997501104816 and parameters: {'alpha': 0.6285416164471327, 'beta': 0.060964303942091916}. Best is trial 82 with value: 0.28874685465159533.


[0.28710890407679845, 0.2875996520624911, 0.28907150788675157, 0.28837918621752523, 0.2893906248116744]
EvaluatorHoldout: Processed 27072 (100.0%) in 11.94 sec. Users per second: 2267
EvaluatorHoldout: Processed 27055 (100.0%) in 11.95 sec. Users per second: 2264
EvaluatorHoldout: Processed 27059 (100.0%) in 11.87 sec. Users per second: 2280
EvaluatorHoldout: Processed 27065 (100.0%) in 12.02 sec. Users per second: 2252
EvaluatorHoldout: Processed 27057 (100.0%) in 12.01 sec. Users per second: 2254


[I 2025-12-26 21:31:47,353] Trial 199 finished with value: 0.26027367707107746 and parameters: {'alpha': 0.672366488809217, 'beta': 0.771447127913863}. Best is trial 82 with value: 0.28874685465159533.


[0.25901480251849196, 0.26024654764440985, 0.2609365910617251, 0.26026645803322995, 0.26090398609753024]


# Da qui inizia il training su URM_all

In [ ]:
return

SyntaxError: 'return' outside function (3438313781.py, line 1)

In [ ]:
best_alpha_test = 0.9874643151475879
best_beta_test = 0.8761805316137236
#Trial 188 finished with value: 0.2909859712486159 and parameters: {'alpha': 0.9874643151475879, 'beta': 0.8761805316137236}.

In [ ]:
recommender_knn_f = ItemKNNCFRecommender(URM_all)
recommender_knn_f.fit(**KNN_params)

In [ ]:
recommender_SLIM_f = SLIMElasticNetRecommender(URM_all)
recommender_SLIM_f.fit(**SLIM_params)

In [ ]:
# Train the final model 
new_similarity = (1 - best_alpha_test) * csr_matrix(recommender_knn_f.W_sparse) + best_alpha_test * recommender_SLIM_f.W_sparse 

hybridrecommender_object_f = ItemKNNCustomSimilarityRecommender(URM_all)
hybridrecommender_object_f.fit(new_similarity)

In [ ]:
als_recommender_f = FeatureCombinedImplicitALSRecommender(URM_all)
als_recommender_f.fit(**IALS_params)

In [ ]:
recommenders = [hybridrecommender_object_f, als_recommender_f]

linear_comb_rec_f = GeneralizedLinearCoupleHybridRecommender(URM_all, recommenders)
linear_comb_rec_f.fit(best_beta_test)

In [ ]:
import time

start_time = time.time()
results = []
for user_id in df_test_user["user_id"].tolist():
        recommendations = linear_comb_rec_f.recommend(user_id, cutoff=20)       
       
        results.append((user_id, ' '.join(map(str, recommendations))))

df_recommendations = pd.DataFrame(results, columns=["user_id", "item_list"])
print(df_recommendations)
df_recommendations.to_csv("recommendations_m2_knn_1.csv", index=False)

end_time = time.time()